# 面试常见代码库

# 字符串与滑动窗口

## 3. 无重复字符的最长子串

In [ ]:
def length_of_longest_substring(text):
    last_position = {}
    left = 0
    longest = 0

    for right in range(len(text)):
        current_character = text[right]

        # 如果字符在当前窗口中出现过，左边界直接跳到重复字符之后。
        if current_character in last_position:
            previous_position = last_position[current_character]
            if previous_position >= left:
                left = previous_position + 1

        last_position[current_character] = right
        current_length = right - left + 1
        longest = max(longest, current_length)

    return longest


input_text = "abcabcbb"
expected_output = 3
actual_output = length_of_longest_substring(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 5. 最长回文子串

In [ ]:
def longest_palindrome(text):
    answer_left = 0
    answer_right = 0
    length = len(text)

    # 一共有 2 * length - 1 个奇数或偶数回文中心。
    for center in range(2 * length - 1):
        left = center // 2
        right = (center + 1) // 2

        while left >= 0 and right < length and text[left] == text[right]:
            left -= 1
            right += 1

        # 循环结束后，[left + 1, right - 1] 是当前回文串。
        current_length = right - left - 1
        answer_length = answer_right - answer_left + 1
        if current_length > answer_length:
            answer_left = left + 1
            answer_right = right - 1

    return text[answer_left:answer_right + 1]


input_text = "babad"
expected_output = "bab 或 aba"
actual_output = longest_palindrome(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 76. 最小覆盖子串

In [ ]:
def is_interval_covered(needed, have):
    # 判断当前窗口是否已覆盖目标字符串中的全部字符
    for character, required_count in needed.items():
        if have.get(character, 0) < required_count:
            return False
    return True


def min_window(source, target):
    # 滑动窗口扩展右边界后，满足条件则收缩到最小
    if not source or not target:
        return ""

    needed = {}
    for character in target:
        needed[character] = needed.get(character, 0) + 1

    have = {}
    left = 0
    best_left = 0
    best_len = len(source) + 1

    for right, character in enumerate(source):
        have[character] = have.get(character, 0) + 1

        while is_interval_covered(needed, have):
            current_len = right - left + 1
            if current_len < best_len:
                best_len = current_len
                best_left = left

            left_character = source[left]
            have[left_character] -= 1
            left += 1

    if best_len > len(source):
        return ""
    return source[best_left:best_left + best_len]


input_data = ("ADOBECODEBANC", "ABC")
expected_output = "BANC"
actual_output = min_window(input_data[0], input_data[1])

print("输入：", input_data)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 数据结构设计

## 146. LRU 缓存

In [ ]:
class DoubleLinkedNode:
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.previous = None
        self.next = None


class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.head = DoubleLinkedNode()
        self.tail = DoubleLinkedNode()
        self.head.next = self.tail
        self.tail.previous = self.head

    def get(self, key):
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self.move_node_to_front(node)
        return node.value

    def put(self, key, value):
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self.move_node_to_front(node)
            return

        new_node = DoubleLinkedNode(key, value)
        self.cache[key] = new_node
        self.add_node_to_head(new_node)

        if len(self.cache) > self.capacity:
            removed_node = self.remove_tail_node()
            del self.cache[removed_node.key]

    def add_node_to_head(self, node):
        node.next = self.head.next
        node.previous = self.head
        self.head.next.previous = node
        self.head.next = node

    def detach_node(self, node):
        node.previous.next = node.next
        node.next.previous = node.previous

    def move_node_to_front(self, node):
        self.detach_node(node)
        self.add_node_to_head(node)

    def remove_tail_node(self):
        node = self.tail.previous
        self.detach_node(node)
        return node


operations = [
    ("put", 1, 1),
    ("put", 2, 2),
    ("get", 1),
    ("put", 3, 3),
    ("get", 2),
]
expected_output = [1, -1]

cache = LRUCache(2)
actual_output = []
for operation in operations:
    if operation[0] == "put":
        cache.put(operation[1], operation[2])
    else:
        actual_output.append(cache.get(operation[1]))

print("输入：capacity = 2, operations =", operations)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 232. 用栈实现队列

In [ ]:
class MyQueue:
    def __init__(self):
        self.input_stack = []
        self.output_stack = []

    def push(self, value): 
        self.input_stack.append(value)

    def pop(self):
        self.transfer_if_needed()
        return self.output_stack.pop()

    def peek(self):
        self.transfer_if_needed()
        return self.output_stack[-1]

    def empty(self):
        return not self.input_stack and not self.output_stack

    def transfer_if_needed(self):
        if not self.output_stack:
            while self.input_stack:
                self.output_stack.append(self.input_stack.pop())


operations = [("push", 1), ("push", 2), ("peek",), ("pop",), ("empty",)]
expected_output = [1, 1, False]

queue = MyQueue()
actual_output = []
for operation in operations:
    if operation[0] == "push":
        queue.push(operation[1])
    elif operation[0] == "peek":
        actual_output.append(queue.peek())
    elif operation[0] == "pop":
        actual_output.append(queue.pop())
    else:
        actual_output.append(queue.empty())

print("输入：", operations)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 155. 最小栈

In [ ]:
class MinStack:
    def __init__(self):
        self.values = []
        self.minimums = []

    def push(self, value):
        self.values.append(value)
        if not self.minimums:
            self.minimums.append(value)
        else:
            self.minimums.append(min(value, self.minimums[-1]))

    def pop(self):
        self.minimums.pop()
        return self.values.pop()

    def top(self):
        return self.values[-1]

    def get_min(self):
        return self.minimums[-1]


stack = MinStack()
input_values = [-2, 0, -3]
for value in input_values:
    stack.push(value)

actual_output = [stack.get_min()]
stack.pop()
actual_output.append(stack.top())
actual_output.append(stack.get_min())
expected_output = [-3, 0, -2]

print("输入：依次 push", input_values, "然后 getMin、pop、top、getMin")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 链表

## 206. 反转链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def reverse_list(head):
    previous = None
    current = head

    while current is not None:
        next_node = current.next
        current.next = previous
        previous = current
        current = next_node

    return previous


input_values = [1, 2, 3, 4, 5]
expected_output = [5, 4, 3, 2, 1]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(reverse_list(head))

print("输入：", input_values)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 25. K 个一组翻转链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def reverse_segment(head):
    previous = None
    current = head
    while current is not None:
        next_node = current.next
        current.next = previous
        previous = current
        current = next_node
    return previous


def reverse_k_group(head, group_size):
    dummy = ListNode(0, head)
    previous_group_tail = dummy
    group_end = dummy

    while group_end.next is not None:
        for _ in range(group_size):
            group_end = group_end.next
            if group_end is None:
                return dummy.next

        group_start = previous_group_tail.next
        next_group_start = group_end.next
        group_end.next = None

        previous_group_tail.next = reverse_segment(group_start)
        group_start.next = next_group_start

        previous_group_tail = group_start
        group_end = previous_group_tail

    return dummy.next


input_values = [1, 2, 3, 4, 5]
group_size = 2
expected_output = [2, 1, 4, 3, 5]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(reverse_k_group(head, group_size))

print("输入：values =", input_values, ", k =", group_size)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 21. 合并两个有序链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def merge_two_lists(list1, list2):
    dummy = ListNode()
    current = dummy

    while list1 is not None and list2 is not None:
        if list1.value < list2.value:
            current.next = list1
            list1 = list1.next
        else:
            current.next = list2
            list2 = list2.next
        current = current.next

    current.next = list1 if list1 is not None else list2
    return dummy.next


input_list1 = [1, 2, 4]
input_list2 = [1, 3, 4]
expected_output = [1, 1, 2, 3, 4, 4]
head1 = build_linked_list(input_list1)
head2 = build_linked_list(input_list2)
actual_output = linked_list_to_list(merge_two_lists(head1, head2))

print("输入：", input_list1, input_list2)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 92. 反转链表 II

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def reverse_list(head):
    previous = None
    current = head
    while current is not None:
        next_node = current.next
        current.next = previous
        previous = current
        current = next_node
    return previous


def reverse_between(head, left, right):
    dummy = ListNode(0, head)

    before_left = dummy
    for _ in range(1, left):
        before_left = before_left.next

    start_node = before_left.next
    end_node = head
    for _ in range(1, right):
        end_node = end_node.next
    after_right = end_node.next

    # 按 Word 原解法切断区间、单独反转，再重新连接。
    before_left.next = None
    end_node.next = None
    reversed_head = reverse_list(start_node)
    before_left.next = reversed_head
    start_node.next = after_right

    return dummy.next


input_values = [1, 2, 3, 4, 5]
left = 2
right = 4
expected_output = [1, 4, 3, 2, 5]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(reverse_between(head, left, right))

print("输入：values =", input_values, ", left =", left, ", right =", right)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 141. 环形链表

In [ ]:
class ListNode:
    def __init__(self, value=0):
        self.value = value
        self.next = None


def build_cycle_list(values, cycle_position):
    nodes = [ListNode(value) for value in values]
    for index in range(len(nodes) - 1):
        nodes[index].next = nodes[index + 1]
    if nodes and cycle_position >= 0:
        nodes[-1].next = nodes[cycle_position]
    return nodes[0] if nodes else None


def has_cycle(head):
    slow = head
    fast = head

    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True

    return False


input_values = [3, 2, 0, -4]
cycle_position = 1
expected_output = True
head = build_cycle_list(input_values, cycle_position)
actual_output = has_cycle(head)

print("输入：values =", input_values, ", pos =", cycle_position)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 143. 重排链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def reorder_list(head):
    if head is None or head.next is None:
        return head

    slow = head
    fast = head
    while fast.next is not None and fast.next.next is not None:
        slow = slow.next
        fast = fast.next.next

    second_half = slow.next
    slow.next = None

    previous = None
    current = second_half
    while current is not None:
        next_node = current.next
        current.next = previous
        previous = current
        current = next_node

    first_half = head
    second_half = previous
    while second_half is not None:
        first_next = first_half.next
        second_next = second_half.next
        first_half.next = second_half
        second_half.next = first_next
        first_half = first_next
        second_half = second_next

    return head


input_values = [1, 2, 3, 4, 5]
expected_output = [1, 5, 2, 4, 3]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(reorder_list(head))

print("输入：", input_values)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 23. 合并 K 个升序链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def merge_two_lists(list1, list2):
    dummy = ListNode()
    current = dummy

    while list1 is not None and list2 is not None:
        if list1.value < list2.value:
            current.next = list1
            list1 = list1.next
        else:
            current.next = list2
            list2 = list2.next
        current = current.next

    current.next = list1 if list1 is not None else list2
    return dummy.next


def merge_k_lists_by_range(lists, left, right):
    # 分治合并：先算中点左右两段，再把两段有序链表拼回
    count = right - left
    if count == 0:
        return None
    if count == 1:
        return lists[left]

    middle = left + count // 2
    left_list = merge_k_lists_by_range(lists, left, middle)
    right_list = merge_k_lists_by_range(lists, middle, right)
    return merge_two_lists(left_list, right_list)


def merge_k_lists(lists):
    return merge_k_lists_by_range(lists, 0, len(lists))


input_lists = [[1, 4, 5], [1, 3, 4], [2, 6]]
expected_output = [1, 1, 2, 3, 4, 4, 5, 6]
heads = [build_linked_list(values) for values in input_lists]
actual_output = linked_list_to_list(merge_k_lists(heads))

print("输入：", input_lists)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 160. 相交链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def get_intersection_node(head_a, head_b):
    pointer_a = head_a
    pointer_b = head_b

    # 两个指针分别走 A+B 和 B+A，因此会同时到达交点或同时到达 None。
    while pointer_a is not pointer_b:
        pointer_a = head_b if pointer_a is None else pointer_a.next
        pointer_b = head_a if pointer_b is None else pointer_b.next

    return pointer_a


shared = ListNode(8, ListNode(4, ListNode(5)))
head_a = ListNode(4, ListNode(1, shared))
head_b = ListNode(5, ListNode(6, ListNode(1, shared)))
expected_output = 8
intersection = get_intersection_node(head_a, head_b)
actual_output = intersection.value if intersection is not None else None

print("输入：A = [4, 1, 8, 4, 5], B = [5, 6, 1, 8, 4, 5]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 82. 删除排序链表中的重复元素 II

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def delete_duplicates(head):
    dummy = ListNode(0, head)
    current = dummy

    while current.next is not None and current.next.next is not None:
        duplicate_value = current.next.value
        if current.next.next.value == duplicate_value:
            while current.next is not None and current.next.value == duplicate_value:
                current.next = current.next.next
        else:
            current = current.next

    return dummy.next


input_values = [1, 2, 3, 3, 4, 4, 5]
expected_output = [1, 2, 5]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(delete_duplicates(head))

print("输入：", input_values)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 19. 删除链表的倒数第 N 个结点

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    fast = dummy
    slow = dummy

    for _ in range(n):
        fast = fast.next

    while fast.next is not None:
        fast = fast.next
        slow = slow.next

    slow.next = slow.next.next
    return dummy.next


input_values = [1, 2, 3, 4, 5]
n = 2
expected_output = [1, 2, 3, 5]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(remove_nth_from_end(head, n))

print("输入：values =", input_values, ", n =", n)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 142. 环形链表 II

In [ ]:
class ListNode:
    def __init__(self, value=0):
        self.value = value
        self.next = None


def build_cycle_list(values, cycle_position):
    nodes = [ListNode(value) for value in values]
    for index in range(len(nodes) - 1):
        nodes[index].next = nodes[index + 1]
    if nodes and cycle_position >= 0:
        nodes[-1].next = nodes[cycle_position]
    return (nodes[0] if nodes else None), nodes


def detect_cycle(head):
    slow = head
    fast = head

    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            break
    else:
        return None

    fast = head
    while fast is not slow:
        fast = fast.next
        slow = slow.next
    return fast


input_values = [3, 2, 0, -4]
cycle_position = 1
expected_output = 1
head, nodes = build_cycle_list(input_values, cycle_position)
entrance = detect_cycle(head)
actual_output = nodes.index(entrance) if entrance is not None else -1

print("输入：values =", input_values, ", pos =", cycle_position)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 148. 排序链表

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def merge_lists(head1, head2):
    dummy = ListNode()
    current = dummy
    while head1 is not None and head2 is not None:
        if head1.value <= head2.value:
            current.next = head1
            head1 = head1.next
        else:
            current.next = head2
            head2 = head2.next
        current = current.next
    current.next = head1 if head1 is not None else head2
    return dummy.next


def sort_list(head):
    if head is None or head.next is None:
        return head

    slow = head
    fast = head
    while fast.next is not None and fast.next.next is not None:
        slow = slow.next
        fast = fast.next.next

    right_head = slow.next
    slow.next = None
    left = sort_list(head)
    right = sort_list(right_head)
    return merge_lists(left, right)


input_values = [4, 2, 1, 3]
expected_output = [1, 2, 3, 4]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(sort_list(head))

print("输入：", input_values)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 2. 两数相加

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def add_two_numbers(list1, list2):
    dummy = ListNode()
    current = dummy
    carry = 0

    while list1 is not None or list2 is not None or carry != 0:
        value1 = list1.value if list1 is not None else 0
        value2 = list2.value if list2 is not None else 0
        total = value1 + value2 + carry

        current.next = ListNode(total % 10)
        current = current.next
        carry = total // 10

        if list1 is not None:
            list1 = list1.next
        if list2 is not None:
            list2 = list2.next

    return dummy.next


input_list1 = [2, 4, 3]
input_list2 = [5, 6, 4]
expected_output = [7, 0, 8]
head1 = build_linked_list(input_list1)
head2 = build_linked_list(input_list2)
actual_output = linked_list_to_list(add_two_numbers(head1, head2))

print("输入：", input_list1, input_list2)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 剑指 Offer 22. 链表中倒数第 k 个节点

In [ ]:
class ListNode:
    def __init__(self, value=0, next_node=None):
        self.value = value
        self.next = next_node


def build_linked_list(values):
    dummy = ListNode()
    current = dummy
    for value in values:
        current.next = ListNode(value)
        current = current.next
    return dummy.next


def linked_list_to_list(head):
    values = []
    while head is not None:
        values.append(head.value)
        head = head.next
    return values


def get_kth_from_end(head, k):
    if head is None or k <= 0:
        return None

    fast = head
    slow = head
    for _ in range(k):
        if fast is None:
            return None
        fast = fast.next

    while fast is not None:
        fast = fast.next
        slow = slow.next
    return slow


input_values = [1, 2, 3, 4, 5]
k = 2
expected_output = [4, 5]
head = build_linked_list(input_values)
actual_output = linked_list_to_list(get_kth_from_end(head, k))

print("输入：values =", input_values, ", k =", k)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 数组、哈希与双指针

## 15. 三数之和

In [ ]:
def three_sum(numbers):
    numbers.sort()
    result = []

    for first in range(len(numbers) - 2):
        if first > 0 and numbers[first] == numbers[first - 1]:
            continue

        if numbers[first] + numbers[first + 1] + numbers[first + 2] > 0:
            break
        if numbers[first] + numbers[-2] + numbers[-1] < 0:
            continue

        second = first + 1
        third = len(numbers) - 1
        while second < third:
            total = numbers[first] + numbers[second] + numbers[third]
            if total < 0:
                second += 1
            elif total > 0:
                third -= 1
            else:
                result.append([numbers[first], numbers[second], numbers[third]])
                second += 1
                third -= 1
                while second < third and numbers[second] == numbers[second - 1]:
                    second += 1
                while second < third and numbers[third] == numbers[third + 1]:
                    third -= 1

    return result


input_numbers = [-1, 0, 1, 2, -1, -4]
expected_output = [[-1, -1, 2], [-1, 0, 1]]
actual_output = three_sum(input_numbers.copy())

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 88. 合并两个有序数组

In [ ]:
def merge_sorted_array(numbers1, valid_count1, numbers2, valid_count2):
    index1 = valid_count1 - 1
    index2 = valid_count2 - 1
    write_index = valid_count1 + valid_count2 - 1

    # 从后向前写入，避免覆盖 numbers1 中尚未处理的元素。
    while index1 >= 0 and index2 >= 0:
        if numbers1[index1] > numbers2[index2]:
            numbers1[write_index] = numbers1[index1]
            index1 -= 1
        else:
            numbers1[write_index] = numbers2[index2]
            index2 -= 1
        write_index -= 1

    while index2 >= 0:
        numbers1[write_index] = numbers2[index2]
        index2 -= 1
        write_index -= 1


input_numbers1 = [1, 2, 3, 0, 0, 0]
input_numbers2 = [2, 5, 6]
expected_output = [1, 2, 2, 3, 5, 6]
actual_output = input_numbers1.copy()
merge_sorted_array(actual_output, 3, input_numbers2, 3)

print("输入：nums1 =", input_numbers1, ", nums2 =", input_numbers2)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 1. 两数之和

In [ ]:
def two_sum(numbers, target):
    value_to_index = {}

    for index in range(len(numbers)):
        required_value = target - numbers[index]
        if required_value in value_to_index:
            return [value_to_index[required_value], index]
        value_to_index[numbers[index]] = index

    return []


input_numbers = [2, 7, 11, 15]
target = 9
expected_output = [0, 1]
actual_output = two_sum(input_numbers, target)

print("输入：numbers =", input_numbers, ", target =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 42. 接雨水

In [ ]:
def trap(height):
    length = len(height)
    prefix_maximum = [0] * length
    suffix_maximum = [0] * length

    maximum = 0
    for index in range(1, length):
        prefix_maximum[index] = max(height[index - 1], maximum)
        maximum = prefix_maximum[index]

    maximum = 0
    for index in range(length - 2, -1, -1):
        suffix_maximum[index] = max(height[index + 1], maximum)
        maximum = suffix_maximum[index]

    water = 0
    for index in range(length):
        current_water = min(prefix_maximum[index], suffix_maximum[index]) - height[index]
        if current_water > 0:
            water += current_water
    return water


input_height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]
expected_output = 6
actual_output = trap(input_height)

print("输入：", input_height)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 41. 缺失的第一个正数

In [ ]:
def first_missing_positive(numbers):
    length = len(numbers)

    for index in range(length):
        while 1 <= numbers[index] <= length:
            correct_index = numbers[index] - 1
            if numbers[correct_index] == numbers[index]:
                break
            numbers[index], numbers[correct_index] = numbers[correct_index], numbers[index]

    for index in range(length):
        if numbers[index] != index + 1:
            return index + 1
    return length + 1


input_numbers = [3, 4, -1, 1]
expected_output = 2
actual_output = first_missing_positive(input_numbers.copy())

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 128. 最长连续序列

In [ ]:
def longest_consecutive(numbers):
    number_set = set(numbers)
    longest = 0

    for number in number_set:
        # 只从一个连续序列的最小值开始向后枚举。
        if number - 1 not in number_set:
            current_number = number
            current_length = 1

            while current_number + 1 in number_set:
                current_number += 1
                current_length += 1

            longest = max(longest, current_length)

    return longest


input_numbers = [100, 4, 200, 1, 3, 2]
expected_output = 4
actual_output = longest_consecutive(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 31. 下一个排列

In [ ]:
def next_permutation(numbers):
    first_decreasing = len(numbers) - 2
    while first_decreasing >= 0 and numbers[first_decreasing] >= numbers[first_decreasing + 1]:
        first_decreasing -= 1

    if first_decreasing >= 0:
        larger_index = len(numbers) - 1
        while numbers[larger_index] <= numbers[first_decreasing]:
            larger_index -= 1
        numbers[first_decreasing], numbers[larger_index] = numbers[larger_index], numbers[first_decreasing]

    left = first_decreasing + 1
    right = len(numbers) - 1
    while left < right:
        numbers[left], numbers[right] = numbers[right], numbers[left]
        left += 1
        right -= 1


input_numbers = [1, 2, 3]
expected_output = [1, 3, 2]
actual_output = input_numbers.copy()
next_permutation(actual_output)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 560. 和为 K 的子数组

In [ ]:
def subarray_sum(numbers, target):
    prefix_count = {0: 1}
    prefix_sum = 0
    answer = 0

    for number in numbers:
        prefix_sum += number
        required_prefix = prefix_sum - target
        answer += prefix_count.get(required_prefix, 0)
        prefix_count[prefix_sum] = prefix_count.get(prefix_sum, 0) + 1

    return answer


input_numbers = [1, 1, 1]
target = 2
expected_output = 2
actual_output = subarray_sum(input_numbers, target)

print("输入：numbers =", input_numbers, ", k =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 排序与选择

## 215. 数组中的第 K 个最大元素

In [ ]:
def swap(numbers, first, second):
    temporary = numbers[first]
    numbers[first] = numbers[second]
    numbers[second] = temporary


def partition(numbers, left, right):
    middle = left + (right - left) // 2
    pivot = numbers[middle]
    swap(numbers, middle, left)

    first = left + 1
    second = right
    while True:
        while first <= second and numbers[first] < pivot:
            first += 1
        while first <= second and numbers[second] > pivot:
            second -= 1
        if first >= second:
            break
        swap(numbers, first, second)
        first += 1
        second -= 1

    swap(numbers, left, second)
    return second


def find_kth_largest(numbers, k):
    target_index = len(numbers) - k
    left = 0
    right = len(numbers) - 1

    while True:
        pivot_index = partition(numbers, left, right)
        if pivot_index == target_index:
            return numbers[pivot_index]
        if pivot_index > target_index:
            right = pivot_index - 1
        else:
            left = pivot_index + 1


input_numbers = [3, 2, 1, 5, 6, 4]
k = 2
expected_output = 5
actual_output = find_kth_largest(input_numbers.copy(), k)

print("输入：numbers =", input_numbers, ", k =", k)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 补充题4. 手撕快速排序

In [ ]:
import random


def quick_sort_swap_elements(numbers, first, second):
    # 快排中的原地交换工具，保持区间元素重排稳定性
    numbers[first], numbers[second] = numbers[second], numbers[first]


def quick_sort_partition_index(numbers, left, right):
    # 以基准值为核心完成一次分区，返回基准最终下标
    random_pivot = random.randint(left, right)
    quick_sort_swap_elements(numbers, random_pivot, left)
    pivot = numbers[left]
    first = left + 1
    second = right

    while True:
        while first <= second and numbers[first] < pivot:
            first += 1
        while first <= second and numbers[second] > pivot:
            second -= 1
        if first >= second:
            break
        quick_sort_swap_elements(numbers, first, second)
        first += 1
        second -= 1

    quick_sort_swap_elements(numbers, left, second)
    return second


def quick_sort_range_recurse(numbers, left, right):
    # 递归处理左右两段子区间
    if left >= right:
        return
    pivot_index = quick_sort_partition_index(numbers, left, right)
    quick_sort_range_recurse(numbers, left, pivot_index - 1)
    quick_sort_range_recurse(numbers, pivot_index + 1, right)


def quick_sort(numbers):
    quick_sort_range_recurse(numbers, 0, len(numbers) - 1)
    return numbers


input_numbers = [5, 2, 3, 1]
expected_output = [1, 2, 3, 5]
random.seed(0)
actual_output = quick_sort(input_numbers.copy())

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 56. 合并区间

In [ ]:
def merge_intervals(intervals):
    intervals.sort(key=lambda interval: interval[0])
    merged = []

    for interval in intervals:
        if not merged or interval[0] > merged[-1][1]:
            merged.append(interval.copy())
        else:
            merged[-1][1] = max(merged[-1][1], interval[1])

    return merged


input_intervals = [[1, 3], [2, 6], [8, 10], [15, 18]]
expected_output = [[1, 6], [8, 10], [15, 18]]
actual_output = merge_intervals([interval.copy() for interval in input_intervals])

print("输入：", input_intervals)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 179. 最大数

In [ ]:
from functools import cmp_to_key


def compare(number1, number2):
    combination1 = number1 + number2
    combination2 = number2 + number1
    if combination1 > combination2:
        return -1
    if combination1 < combination2:
        return 1
    return 0


def largest_number(numbers):
    string_numbers = [str(number) for number in numbers]
    string_numbers.sort(key=cmp_to_key(compare))
    if string_numbers[0] == "0":
        return "0"
    return "".join(string_numbers)


input_numbers = [10, 2]
expected_output = "210"
actual_output = largest_number(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 补充题6. 手撕堆排序

In [ ]:
def sift_down_heap(numbers, parent, heap_size):
    # 从父节点向下调整，恢复堆性质并完成一次下沉
    while 2 * parent + 1 < heap_size:
        child = 2 * parent + 1
        if child + 1 < heap_size and numbers[child + 1] > numbers[child]:
            child += 1
        if numbers[parent] >= numbers[child]:
            break
        numbers[parent], numbers[child] = numbers[child], numbers[parent]
        parent = child


def heap_sort(numbers):
    for parent in range(len(numbers) // 2 - 1, -1, -1):
        sift_down_heap(numbers, parent, len(numbers))

    for end in range(len(numbers) - 1, 0, -1):
        numbers[0], numbers[end] = numbers[end], numbers[0]
        sift_down_heap(numbers, 0, end)

    return numbers


input_numbers = [12, 11, 13, 5, 6, 7]
expected_output = [5, 6, 7, 11, 12, 13]
actual_output = heap_sort(input_numbers.copy())

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 二分查找

## 33. 搜索旋转排序数组

In [ ]:
def search_rotated_array(numbers, target):
    left = 0
    right = len(numbers) - 1

    while left <= right:
        middle = left + (right - left) // 2
        if numbers[middle] == target:
            return middle

        # 注意这里必须是 <=，这样只有两个元素时也能正确判断左半段有序。
        if numbers[left] <= numbers[middle]:
            if numbers[left] <= target < numbers[middle]:
                right = middle - 1
            else:
                left = middle + 1
        else:
            if numbers[middle] < target <= numbers[right]:
                left = middle + 1
            else:
                right = middle - 1

    return -1


input_numbers = [4, 5, 6, 7, 0, 1, 2]
target = 0
expected_output = 4
actual_output = search_rotated_array(input_numbers, target)

print("输入：numbers =", input_numbers, ", target =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 4. 寻找两个正序数组的中位数

In [ ]:
def find_median_sorted_arrays(numbers1, numbers2):
    if len(numbers1) > len(numbers2):
        numbers1, numbers2 = numbers2, numbers1

    length1 = len(numbers1)
    length2 = len(numbers2)
    left = 0
    right = length1
    left_part_size = (length1 + length2 + 1) // 2

    while left <= right:
        partition1 = (left + right) // 2
        partition2 = left_part_size - partition1

        max_left1 = float("-inf") if partition1 == 0 else numbers1[partition1 - 1]
        min_right1 = float("inf") if partition1 == length1 else numbers1[partition1]
        max_left2 = float("-inf") if partition2 == 0 else numbers2[partition2 - 1]
        min_right2 = float("inf") if partition2 == length2 else numbers2[partition2]

        if max_left1 <= min_right2 and max_left2 <= min_right1:
            if (length1 + length2) % 2 == 1:
                return float(max(max_left1, max_left2))
            left_maximum = max(max_left1, max_left2)
            right_minimum = min(min_right1, min_right2)
            return (left_maximum + right_minimum) / 2.0

        if max_left1 > min_right2:
            right = partition1 - 1
        else:
            left = partition1 + 1

    raise ValueError("输入数组必须有序")


input_data = ([1, 3], [2])
expected_output = 2.0
actual_output = find_median_sorted_arrays(input_data[0], input_data[1])

print("输入：", input_data)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 704. 二分查找

In [ ]:
def binary_search(numbers, target):
    left = 0
    right = len(numbers) - 1

    while left <= right:
        middle = left + (right - left) // 2
        if numbers[middle] == target:
            return middle
        if numbers[middle] < target:
            left = middle + 1
        else:
            right = middle - 1

    return -1


input_numbers = [-1, 0, 3, 5, 9, 12]
target = 9
expected_output = 4
actual_output = binary_search(input_numbers, target)

print("输入：numbers =", input_numbers, ", target =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 69. x 的平方根

In [ ]:
def integer_square_root(number):
    if number == 0 or number == 1:
        return number

    left = 1
    right = number
    answer = 0

    while left <= right:
        middle = left + (right - left) // 2
        if middle <= number // middle:
            answer = middle
            left = middle + 1
        else:
            right = middle - 1

    return answer


input_number = 8
expected_output = 2
actual_output = integer_square_root(input_number)

print("输入：", input_number)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 34. 在排序数组中查找元素的第一个和最后一个位置

In [ ]:
def search_range(numbers, target):
    result = [-1, -1]
    left = 0
    right = len(numbers) - 1
    found_position = -1

    while left <= right:
        middle = left + (right - left) // 2
        if numbers[middle] == target:
            found_position = middle
            break
        if numbers[middle] < target:
            left = middle + 1
        else:
            right = middle - 1

    if found_position == -1:
        return result

    start = found_position
    while start > 0 and numbers[start - 1] == target:
        start -= 1

    end = found_position
    while end < len(numbers) - 1 and numbers[end + 1] == target:
        end += 1

    result[0] = start
    result[1] = end
    return result


input_numbers = [5, 7, 7, 8, 8, 10]
target = 8
expected_output = [3, 4]
actual_output = search_range(input_numbers, target)

print("输入：numbers =", input_numbers, ", target =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 162. 寻找峰值

In [ ]:
def find_peak_element(numbers):
    left = -1
    right = len(numbers) - 1

    # 在开区间 (left, right) 中二分。
    while left + 1 < right:
        middle = left + (right - left) // 2
        if numbers[middle] > numbers[middle + 1]:
            right = middle
        else:
            left = middle

    return right


input_numbers = [1, 2, 3, 1]
expected_output = 2
actual_output = find_peak_element(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 动态规划

## 121. 买卖股票的最佳时机

In [ ]:
def max_profit(prices):
    minimum_price = prices[0]
    maximum_profit = 0

    for price in prices:
        minimum_price = min(minimum_price, price)
        current_profit = price - minimum_price
        maximum_profit = max(maximum_profit, current_profit)

    return maximum_profit


input_prices = [7, 1, 5, 3, 6, 4]
expected_output = 5
actual_output = max_profit(input_prices)

print("输入：", input_prices)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 300. 最长递增子序列

In [ ]:
def length_of_lis(numbers):
    if not numbers:
        return 0

    # dp[i] 表示以 numbers[i] 结尾的最长严格递增子序列长度。
    dp = [1] * len(numbers)
    maximum_length = 1

    for current in range(len(numbers)):
        for previous in range(current):
            if numbers[previous] < numbers[current]:
                dp[current] = max(dp[current], dp[previous] + 1)
        maximum_length = max(maximum_length, dp[current])

    return maximum_length


input_numbers = [10, 9, 2, 5, 3, 7, 101, 18]
expected_output = 4
actual_output = length_of_lis(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 72. 编辑距离

In [ ]:
def edit_distance_recursive(word1, word2, index1, index2, memo):
    # 自顶向下记忆化递归：三种编辑操作取最小值
    if index1 < 0:
        return index2 + 1
    if index2 < 0:
        return index1 + 1
    if memo[index1][index2] != -1:
        return memo[index1][index2]

    if word1[index1] == word2[index2]:
        answer = edit_distance_recursive(word1, word2, index1 - 1, index2 - 1, memo)
    else:
        delete_cost = edit_distance_recursive(word1, word2, index1 - 1, index2, memo)
        insert_cost = edit_distance_recursive(word1, word2, index1, index2 - 1, memo)
        replace_cost = edit_distance_recursive(word1, word2, index1 - 1, index2 - 1, memo)
        answer = min(delete_cost, insert_cost, replace_cost) + 1

    memo[index1][index2] = answer
    return answer


def edit_distance(word1, word2):
    memo = [[-1] * len(word2) for _ in range(len(word1))]
    return edit_distance_recursive(word1, word2, len(word1) - 1, len(word2) - 1, memo)


input_data = ("horse", "ros")
expected_output = 3
actual_output = edit_distance(input_data[0], input_data[1])

print("输入：", input_data)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 1143. 最长公共子序列

In [ ]:
def longest_common_subsequence(text1, text2):
    dp = [[0] * (len(text2) + 1) for _ in range(len(text1) + 1)]

    for index1 in range(len(text1)):
        for index2 in range(len(text2)):
            if text1[index1] == text2[index2]:
                dp[index1 + 1][index2 + 1] = dp[index1][index2] + 1
            else:
                dp[index1 + 1][index2 + 1] = max(
                    dp[index1][index2 + 1],
                    dp[index1 + 1][index2],
                )

    return dp[-1][-1]


input_data = ("abcde", "ace")
expected_output = 3
actual_output = longest_common_subsequence(input_data[0], input_data[1])

print("输入：", input_data)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 322. 零钱兑换

In [ ]:
def coin_change(coins, amount):
    impossible = amount + 1
    dp = [impossible] * (amount + 1)
    dp[0] = 0

    for current_amount in range(1, amount + 1):
        for coin in coins:
            if coin <= current_amount:
                dp[current_amount] = min(
                    dp[current_amount],
                    dp[current_amount - coin] + 1,
                )

    return -1 if dp[amount] == impossible else dp[amount]


input_coins = [1, 2, 5]
amount = 11
expected_output = 3
actual_output = coin_change(input_coins, amount)

print("输入：coins =", input_coins, ", amount =", amount)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 64. 最小路径和

In [ ]:
def minimum_path_sum(grid):
    rows = len(grid)
    columns = len(grid[0])
    dp = [[0] * columns for _ in range(rows)]

    dp[0][0] = grid[0][0]
    for column in range(1, columns):
        dp[0][column] = dp[0][column - 1] + grid[0][column]
    for row in range(1, rows):
        dp[row][0] = dp[row - 1][0] + grid[row][0]

    for row in range(1, rows):
        for column in range(1, columns):
            dp[row][column] = min(dp[row - 1][column], dp[row][column - 1])
            dp[row][column] += grid[row][column]

    return dp[-1][-1]


input_grid = [[1, 3, 1], [1, 5, 1], [4, 2, 1]]
expected_output = 7
actual_output = minimum_path_sum(input_grid)

print("输入：", input_grid)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 122. 买卖股票的最佳时机 II

In [ ]:
def max_profit_multiple_transactions(prices):
    total_profit = 0
    for index in range(1, len(prices)):
        if prices[index] > prices[index - 1]:
            total_profit += prices[index] - prices[index - 1]
    return total_profit


input_prices = [7, 1, 5, 3, 6, 4]
expected_output = 7
actual_output = max_profit_multiple_transactions(input_prices)

print("输入：", input_prices)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 221. 最大正方形

In [ ]:
def maximal_square(matrix):
    rows = len(matrix)
    columns = len(matrix[0])
    dp = [[0] * (columns + 1) for _ in range(rows + 1)]
    maximum_side = 0

    for row in range(1, rows + 1):
        for column in range(1, columns + 1):
            if matrix[row - 1][column - 1] == "1":
                dp[row][column] = 1 + min(
                    dp[row - 1][column],
                    dp[row][column - 1],
                    dp[row - 1][column - 1],
                )
                maximum_side = max(maximum_side, dp[row][column])

    return maximum_side * maximum_side


input_matrix = [
    ["1", "0", "1", "0", "0"],
    ["1", "0", "1", "1", "1"],
    ["1", "1", "1", "1", "1"],
    ["1", "0", "0", "1", "0"],
]
expected_output = 4
actual_output = maximal_square(input_matrix)

print("输入：", input_matrix)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 152. 乘积最大子数组

In [ ]:
def maximum_product_subarray(numbers):
    current_maximum = numbers[0]
    current_minimum = numbers[0]
    answer = numbers[0]

    for index in range(1, len(numbers)):
        number = numbers[index]
        previous_maximum = current_maximum
        previous_minimum = current_minimum

        current_maximum = max(number, previous_maximum * number, previous_minimum * number)
        current_minimum = min(number, previous_maximum * number, previous_minimum * number)
        answer = max(answer, current_maximum)

    return answer


input_numbers = [2, 3, -2, 4]
expected_output = 6
actual_output = maximum_product_subarray(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 139. 单词拆分

In [ ]:
def word_break(text, word_dictionary):
    can_split = [False] * (len(text) + 1)
    can_split[0] = True

    for end in range(1, len(text) + 1):
        for start in range(end):
            if can_split[start] and text[start:end] in word_dictionary:
                can_split[end] = True
                break

    return can_split[-1]


input_text = "leetcode"
input_dictionary = ["leet", "code"]
expected_output = True
actual_output = word_break(input_text, input_dictionary)

print("输入：text =", input_text, ", dictionary =", input_dictionary)
print("预期输出：", expected_output)
print("实际输出：", actual_output)

# ---- 解法二：词长剪枝版（DP） ----

def word_break_with_length_pruning(text, word_dictionary):
    word_set = set(word_dictionary)
    minimum_length = min(len(word) for word in word_dictionary)
    maximum_length = max(len(word) for word in word_dictionary)
    can_split = [False] * (len(text) + 1)
    can_split[0] = True

    for end in range(1, len(text) + 1):
        first_start = max(0, end - maximum_length)
        last_start = end - minimum_length

        for start in range(first_start, last_start + 1):
            if can_split[start] and text[start:end] in word_set:
                can_split[end] = True
                break

    return can_split[-1]


input_text = "applepenapple"
input_dictionary = ["apple", "pen"]
expected_output = True
actual_output = word_break_with_length_pruning(input_text, input_dictionary)

print("输入：text =", input_text, ", dictionary =", input_dictionary)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 718. 最长重复子数组

In [ ]:
def find_length(numbers1, numbers2):
    dp = [[0] * (len(numbers2) + 1) for _ in range(len(numbers1) + 1)]
    answer = 0

    for index1 in range(len(numbers1)):
        for index2 in range(len(numbers2)):
            if numbers1[index1] == numbers2[index2]:
                dp[index1 + 1][index2 + 1] = dp[index1][index2] + 1
                answer = max(answer, dp[index1 + 1][index2 + 1])

    return answer


input_numbers1 = [1, 2, 3, 2, 1]
input_numbers2 = [3, 2, 1, 4, 7]
expected_output = 3
actual_output = find_length(input_numbers1, input_numbers2)

print("输入：", input_numbers1, input_numbers2)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 回溯

## 46. 全排列

In [ ]:
def backtrack_all_permutations(numbers, path, used, result):
    # 回溯生成：按位尝试所有未使用数字，构造全排列
    if len(path) == len(numbers):
        result.append(path.copy())
        return

    for index in range(len(numbers)):
        if used[index]:
            continue

        path.append(numbers[index])
        used[index] = True
        backtrack_all_permutations(numbers, path, used, result)
        path.pop()
        used[index] = False


def permutations(numbers):
    result = []
    path = []
    used = [False] * len(numbers)

    backtrack_all_permutations(numbers, path, used, result)
    return result


input_numbers = [1, 2, 3]
expected_output = [
    [1, 2, 3], [1, 3, 2],
    [2, 1, 3], [2, 3, 1],
    [3, 1, 2], [3, 2, 1],
]
actual_output = permutations(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 93. 复原 IP 地址

In [ ]:
def backtrack_ip_addresses(text, start, segment_index, path, result):
    # 回溯枚举 4 段 IP：固定每段范围并剪枝非法前导零
    remaining_characters = len(text) - start
    remaining_segments = 4 - segment_index
    if remaining_characters < remaining_segments:
        return
    if remaining_characters > remaining_segments * 3:
        return

    if start == len(text):
        result.append(".".join(path))
        return

    segment_value = 0
    for end in range(start, min(start + 3, len(text))):
        segment_value = segment_value * 10 + int(text[end])
        if segment_value > 255:
            break

        path[segment_index] = text[start:end + 1]
        backtrack_ip_addresses(text, end + 1, segment_index + 1, path, result)

        if segment_value == 0:
            break


def restore_ip_addresses(text):
    result = []
    path = [""] * 4
    backtrack_ip_addresses(text, 0, 0, path, result)
    return result


input_text = "25525511135"
expected_output = ["255.255.11.135", "255.255.111.35"]
actual_output = restore_ip_addresses(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 22. 括号生成

In [ ]:
def backtrack_generate_parentheses(pair_count, left_count, right_count, path, result):
    # 回溯策略：任意时刻左括号/右括号可用性决定下一步选择
    if left_count == pair_count and right_count == pair_count:
        result.append("".join(path))
        return

    if left_count < pair_count:
        path.append("(")
        backtrack_generate_parentheses(pair_count, left_count + 1, right_count, path, result)
        path.pop()

    if right_count < left_count:
        path.append(")")
        backtrack_generate_parentheses(pair_count, left_count, right_count + 1, path, result)
        path.pop()


def generate_parentheses(pair_count):
    result = []
    path = []

    backtrack_generate_parentheses(pair_count, 0, 0, path, result)
    return result


input_pair_count = 3
expected_output = ["((()))", "(()())", "(())()", "()(())", "()()()"]
actual_output = generate_parentheses(input_pair_count)

print("输入：", input_pair_count)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 78. 子集

In [ ]:
def backtrack_subsets(numbers, start, path, result):
    # 回溯枚举：对每个元素决策“选或不选”形成子集
    result.append(path.copy())

    for index in range(start, len(numbers)):
        path.append(numbers[index])
        backtrack_subsets(numbers, index + 1, path, result)
        path.pop()


def subsets(numbers):
    result = []
    path = []

    backtrack_subsets(numbers, 0, path, result)
    return result


input_numbers = [1, 2]
expected_output = [[], [1], [1, 2], [2]]
actual_output = subsets(input_numbers)

print("输入：", input_numbers)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 39. 组合总和

In [ ]:
def backtrack_combination_sum(candidates, index, remaining, path, result):
    # 回溯 + 索引控制：可重复用当前元素或跳到下一个
    if remaining == 0:
        result.append(path.copy())
        return
    if index == len(candidates) or remaining < candidates[index]:
        return

    # 不选择当前数字。
    backtrack_combination_sum(candidates, index + 1, remaining, path, result)

    # 选择当前数字；下次仍可继续选择它。
    path.append(candidates[index])
    backtrack_combination_sum(candidates, index, remaining - candidates[index], path, result)
    path.pop()


def combination_sum(candidates, target):
    candidates.sort()
    result = []
    path = []

    backtrack_combination_sum(candidates, 0, target, path, result)
    return result


input_candidates = [2, 3, 6, 7]
target = 7
expected_output = [[7], [2, 2, 3]]
actual_output = combination_sum(input_candidates.copy(), target)

print("输入：candidates =", input_candidates, ", target =", target)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 栈与单调结构

## 20. 有效的括号

In [ ]:
def is_valid_parentheses(text):
    stack = []
    matching_left = {")": "(", "]": "[", "}": "{"}

    for character in text:
        if character in ("(", "[", "{"):
            stack.append(character)
        else:
            if not stack:
                return False
            if stack.pop() != matching_left[character]:
                return False

    return not stack


input_text = "()[]{}"
expected_output = True
actual_output = is_valid_parentheses(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 239. 滑动窗口最大值

In [ ]:
from collections import deque


def maximum_sliding_window(numbers, window_size):
    decreasing_queue = deque()
    result = []

    for right in range(len(numbers)):
        while decreasing_queue and numbers[decreasing_queue[-1]] <= numbers[right]:
            decreasing_queue.pop()
        decreasing_queue.append(right)

        left = right - window_size + 1
        if decreasing_queue[0] < left:
            decreasing_queue.popleft()

        if left >= 0:
            result.append(numbers[decreasing_queue[0]])

    return result


input_numbers = [1, 3, -1, -3, 5, 3, 6, 7]
window_size = 3
expected_output = [3, 3, 5, 5, 6, 7]
actual_output = maximum_sliding_window(input_numbers, window_size)

print("输入：numbers =", input_numbers, ", k =", window_size)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 32. 最长有效括号

In [ ]:
def longest_valid_parentheses(text):
    stack = [-1]
    longest = 0

    for index in range(len(text)):
        if text[index] == "(":
            stack.append(index)
        else:
            stack.pop()
            if not stack:
                stack.append(index)
            else:
                current_length = index - stack[-1]
                longest = max(longest, current_length)

    return longest


input_text = ")()())"
expected_output = 4
actual_output = longest_valid_parentheses(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 394. 字符串解码

In [ ]:
def decode_string(text):
    string_stack = []
    number_stack = []
    current_number = 0
    current_string = ""

    for character in text:
        if character.isdigit():
            current_number = current_number * 10 + int(character)
        elif character == "[":
            number_stack.append(current_number)
            string_stack.append(current_string)
            current_number = 0
            current_string = ""
        elif character == "]":
            repeat_count = number_stack.pop()
            previous_string = string_stack.pop()
            current_string = previous_string + current_string * repeat_count
        else:
            current_string += character

    return current_string


input_text = "3[a2[c]]"
expected_output = "accaccacc"
actual_output = decode_string(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)

# ---- 解法二：递归版 ----

def decode_range_recursive(text, start, end):
    # 递归按区间解码：处理括号内重复片段并展开
    result = ""
    index = start

    while index < end:
        if text[index].isalpha():
            result += text[index]
            index += 1
            continue

        repeat_count = 0
        while index < end and text[index].isdigit():
            repeat_count = repeat_count * 10 + int(text[index])
            index += 1

        inside_start = index + 1
        bracket_count = 1
        index += 1
        while bracket_count > 0:
            if text[index] == "[":
                bracket_count += 1
            elif text[index] == "]":
                bracket_count -= 1
            index += 1

        inside = decode_range_recursive(text, inside_start, index - 1)
        result += inside * repeat_count

    return result


def decode_string_recursively(text):
    return decode_range_recursive(text, 0, len(text))


input_text = "2[abc]3[cd]ef"
expected_output = "abcabccdcdcdef"
actual_output = decode_string_recursively(input_text)

print("输入：", input_text)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 227. 基本计算器 II

In [ ]:
def calculate(expression):
    stack = []
    current_number = 0
    previous_operator = "+"

    for index in range(len(expression) + 1):
        character = expression[index] if index < len(expression) else "+"

        if character.isdigit():
            current_number = current_number * 10 + int(character)

        if (not character.isdigit() and character != " ") or index == len(expression):
            if previous_operator == "+":
                stack.append(current_number)
            elif previous_operator == "-":
                stack.append(-current_number)
            elif previous_operator == "*":
                stack.append(stack.pop() * current_number)
            else:
                stack.append(int(stack.pop() / current_number))

            previous_operator = character
            current_number = 0

    return sum(stack)


input_expression = "3+2*2"
expected_output = 7
actual_output = calculate(input_expression)

print("输入：", input_expression)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 矩阵与图搜索

## 200. 岛屿数量

In [ ]:
def explore_island_cells(grid, rows, columns, row, column):
    # DFS 标记并扩展一个岛屿的全部陆地单元
    if row < 0 or row >= rows or column < 0 or column >= columns:
        return
    if grid[row][column] == "0":
        return

    grid[row][column] = "0"
    explore_island_cells(grid, rows, columns, row - 1, column)
    explore_island_cells(grid, rows, columns, row + 1, column)
    explore_island_cells(grid, rows, columns, row, column - 1)
    explore_island_cells(grid, rows, columns, row, column + 1)


def number_of_islands(grid):
    rows = len(grid)
    columns = len(grid[0])
    island_count = 0

    for row in range(rows):
        for column in range(columns):
            if grid[row][column] == "1":
                island_count += 1
                explore_island_cells(grid, rows, columns, row, column)

    return island_count


input_grid = [
    ["1", "1", "1", "1", "0"],
    ["1", "1", "0", "1", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "0", "0", "0"],
]
expected_output = 1
grid_copy = [row.copy() for row in input_grid]
actual_output = number_of_islands(grid_copy)

print("输入：", input_grid)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 54. 螺旋矩阵

In [ ]:
def spiral_order(matrix):
    result = []
    top = 0
    bottom = len(matrix) - 1
    left = 0
    right = len(matrix[0]) - 1

    while top <= bottom and left <= right:
        for column in range(left, right + 1):
            result.append(matrix[top][column])
        top += 1

        for row in range(top, bottom + 1):
            result.append(matrix[row][right])
        right -= 1

        if top <= bottom:
            for column in range(right, left - 1, -1):
                result.append(matrix[bottom][column])
            bottom -= 1

        if left <= right:
            for row in range(bottom, top - 1, -1):
                result.append(matrix[row][left])
            left += 1

    return result


input_matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
expected_output = [1, 2, 3, 6, 9, 8, 7, 4, 5]
actual_output = spiral_order(input_matrix)

print("输入：", input_matrix)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 48. 旋转图像

In [ ]:
def rotate_matrix(matrix):
    size = len(matrix)

    # 与 Word 中 Java 代码一致：每次原地轮换四个对称位置。
    for column in range(size // 2):
        for row in range((size + 1) // 2):
            temporary = matrix[row][column]
            matrix[row][column] = matrix[size - column - 1][row]
            matrix[size - column - 1][row] = matrix[size - row - 1][size - column - 1]
            matrix[size - row - 1][size - column - 1] = matrix[column][size - row - 1]
            matrix[column][size - row - 1] = temporary


input_matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
expected_output = [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
actual_output = [row.copy() for row in input_matrix]
rotate_matrix(actual_output)

print("输入：", input_matrix)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# 二叉树

## 102. 二叉树的层序遍历

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def collect_level_order_nodes(node, level, result):
    # DFS 中按层累积节点，辅助层序遍历结果拼接
    if node is None:
        return

    if len(result) == level:
        result.append([])
    result[level].append(node.value)

    collect_level_order_nodes(node.left, level + 1, result)
    collect_level_order_nodes(node.right, level + 1, result)


def level_order(root):
    result = []
    collect_level_order_nodes(root, 0, result)
    return result


root = TreeNode(3)
root.left = TreeNode(9)
root.right = TreeNode(20, TreeNode(15), TreeNode(7))
expected_output = [[3], [9, 20], [15, 7]]
actual_output = level_order(root)

print("输入：层序 [3, 9, 20, null, null, 15, 7]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 236. 二叉树的最近公共祖先

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def lowest_common_ancestor(root, node_p, node_q):
    if root is None or root is node_p or root is node_q:
        return root

    left_result = lowest_common_ancestor(root.left, node_p, node_q)
    right_result = lowest_common_ancestor(root.right, node_p, node_q)

    if left_result is not None and right_result is not None:
        return root
    if left_result is not None:
        return left_result
    return right_result


node5 = TreeNode(5)
node1 = TreeNode(1)
root = TreeNode(3, node5, node1)
node5.left = TreeNode(6)
node5.right = TreeNode(2, TreeNode(7), TreeNode(4))
node1.left = TreeNode(0)
node1.right = TreeNode(8)

expected_output = 3
ancestor = lowest_common_ancestor(root, node5, node1)
actual_output = ancestor.value

print("输入：层序 [3,5,1,6,2,0,8,null,null,7,4], p = 5, q = 1")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 103. 二叉树的锯齿形层序遍历

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def traverse_zigzag_levels(node, level, result):
    # DFS 先序遍历并按层反转奇偶层实现锯齿顺序
    if node is None:
        return

    while len(result) <= level:
        result.append([])

    if level % 2 == 0:
        result[level].append(node.value)
    else:
        result[level].insert(0, node.value)

    traverse_zigzag_levels(node.left, level + 1, result)
    traverse_zigzag_levels(node.right, level + 1, result)


def zigzag_level_order(root):
    result = []
    traverse_zigzag_levels(root, 0, result)
    return result


root = TreeNode(3)
root.left = TreeNode(9)
root.right = TreeNode(20, TreeNode(15), TreeNode(7))
expected_output = [[3], [20, 9], [15, 7]]
actual_output = zigzag_level_order(root)

print("输入：层序 [3, 9, 20, null, null, 15, 7]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 124. 二叉树中的最大路径和

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def calculate_max_path_sum(node, best):
    # 后序 DFS 计算包含当前节点的最大链式路径并更新全局最优
    if node is None:
        return 0

    left_sum = max(calculate_max_path_sum(node.left, best), 0)
    right_sum = max(calculate_max_path_sum(node.right, best), 0)

    path_through_node = node.value + left_sum + right_sum
    best[0] = max(best[0], path_through_node)

    return node.value + max(left_sum, right_sum)


def maximum_path_sum(root):
    best = [float("-inf")]
    calculate_max_path_sum(root, best)
    return best[0]


root = TreeNode(-10)
root.left = TreeNode(9)
root.right = TreeNode(20, TreeNode(15), TreeNode(7))
expected_output = 42
actual_output = maximum_path_sum(root)

print("输入：层序 [-10, 9, 20, null, null, 15, 7]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 199. 二叉树的右视图

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def collect_right_side_view(node, level, result):
    # DFS 优先右子树，首次到达每层时记录右视图节点
    if node is None:
        return

    # 先访问右子树，因此每层首次访问的节点就是右视图节点。
    if level == len(result):
        result.append(node.value)

    collect_right_side_view(node.right, level + 1, result)
    collect_right_side_view(node.left, level + 1, result)


def right_side_view(root):
    result = []
    collect_right_side_view(root, 0, result)
    return result


root = TreeNode(1)
root.left = TreeNode(2, None, TreeNode(5))
root.right = TreeNode(3, None, TreeNode(4))
expected_output = [1, 3, 4]
actual_output = right_side_view(root)

print("输入：层序 [1, 2, 3, null, 5, null, 4]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 94. 二叉树的中序遍历

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def inorder_traversal(root):
    result = []
    stack = []
    current = root

    while current is not None or stack:
        while current is not None:
            stack.append(current)
            current = current.left

        current = stack.pop()
        result.append(current.value)
        current = current.right

    return result


root = TreeNode(1)
root.right = TreeNode(2, TreeNode(3), None)
expected_output = [1, 3, 2]
actual_output = inorder_traversal(root)

print("输入：层序 [1, null, 2, 3]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 105. 从前序与中序遍历序列构造二叉树

In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def build_tree_from_preorder_range(preorder, inorder_left, inorder_right, value_to_inorder_index, preorder_index):
    # 利用中序索引表定位根节点位置，递归构建左右子树
    if inorder_left > inorder_right:
        return None

    root_value = preorder[preorder_index[0]]
    preorder_index[0] += 1
    root = TreeNode(root_value)

    inorder_root_index = value_to_inorder_index[root_value]
    root.left = build_tree_from_preorder_range(preorder, inorder_left, inorder_root_index - 1, value_to_inorder_index, preorder_index)
    root.right = build_tree_from_preorder_range(preorder, inorder_root_index + 1, inorder_right, value_to_inorder_index, preorder_index)
    return root


def build_tree(preorder, inorder):
    value_to_inorder_index = {}
    for index, value in enumerate(inorder):
        value_to_inorder_index[value] = index

    preorder_index = [0]
    return build_tree_from_preorder_range(preorder, 0, len(inorder) - 1, value_to_inorder_index, preorder_index)


def tree_to_level_order(root):
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
        else:
            result.append(node.value)
            queue.append(node.left)
            queue.append(node.right)
    while result and result[-1] is None:
        result.pop()
    return result


input_preorder = [3, 9, 20, 15, 7]
input_inorder = [9, 3, 15, 20, 7]
expected_output = [3, 9, 20, None, None, 15, 7]
root = build_tree(input_preorder, input_inorder)
actual_output = tree_to_level_order(root)

print("输入：preorder =", input_preorder, ", inorder =", input_inorder)
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 129. 求根节点到叶节点数字之和

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def dfs_sum_root_to_leaf_numbers(node, current_number):
    # 沿路径累加位权数字，叶子节点把单条路径值计入答案
    if node is None:
        return 0

    current_number = current_number * 10 + node.value
    if node.left is None and node.right is None:
        return current_number

    left_sum = dfs_sum_root_to_leaf_numbers(node.left, current_number)
    right_sum = dfs_sum_root_to_leaf_numbers(node.right, current_number)
    return left_sum + right_sum


def sum_root_to_leaf_numbers(root):
    return dfs_sum_root_to_leaf_numbers(root, 0)


root = TreeNode(1, TreeNode(2), TreeNode(3))
expected_output = 25
actual_output = sum_root_to_leaf_numbers(root)

print("输入：层序 [1, 2, 3]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 101. 对称二叉树

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def is_symmetric_pair(left_node, right_node):
    # 镜像比较：比较左右子树的外侧与内侧节点是否对称
    if left_node is None and right_node is None:
        return True
    if left_node is None or right_node is None:
        return False
    if left_node.value != right_node.value:
        return False

    outside_equal = is_symmetric_pair(left_node.left, right_node.right)
    inside_equal = is_symmetric_pair(left_node.right, right_node.left)
    return outside_equal and inside_equal


def is_symmetric(root):
    if root is None:
        return True
    return is_symmetric_pair(root.left, root.right)


root = TreeNode(1)
root.left = TreeNode(2, TreeNode(3), TreeNode(4))
root.right = TreeNode(2, TreeNode(4), TreeNode(3))
expected_output = True
actual_output = is_symmetric(root)

print("输入：层序 [1, 2, 2, 3, 4, 4, 3]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 104. 二叉树的最大深度

In [ ]:
class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def maximum_depth(root):
    if root is None:
        return 0

    left_depth = maximum_depth(root.left)
    right_depth = maximum_depth(root.right)
    return 1 + max(left_depth, right_depth)


root = TreeNode(3)
root.left = TreeNode(9)
root.right = TreeNode(20, TreeNode(15), TreeNode(7))
expected_output = 3
actual_output = maximum_depth(root)

print("输入：层序 [3, 9, 20, null, null, 15, 7]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


## 662. 二叉树最大宽度

In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, value=0, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def width_of_binary_tree(root):
    # BFS 逐层遍历，记录同层首尾索引并计算最大宽度
    if root is None:
        return 0

    maximum_width = 0
    queue = deque([(root, 0)])

    while queue:
        level_size = len(queue)
        first_index = queue[0][1]
        last_normalized_index = 0

        for _ in range(level_size):
            node, position = queue.popleft()
            normalized_position = position - first_index
            last_normalized_index = normalized_position

            if node.left is not None:
                queue.append((node.left, normalized_position * 2))
            if node.right is not None:
                queue.append((node.right, normalized_position * 2 + 1))

        current_width = last_normalized_index + 1
        maximum_width = max(maximum_width, current_width)

    return maximum_width


root = TreeNode(1)
root.left = TreeNode(3, TreeNode(5), TreeNode(3))
root.right = TreeNode(2, None, TreeNode(9))
expected_output = 4
actual_output = width_of_binary_tree(root)

print("输入：层序 [1, 3, 2, 5, 3, null, 9]")
print("预期输出：", expected_output)
print("实际输出：", actual_output)


# Attention

## GroupQueryAttention

## MultiHeadAttention

## MultiLatentAttention

## ScaledDotProductAttention

# Normalization

## LayerNorm

## RMSNorm

# Position Encoding

## RotaryEmbedding

# FFN

## FFN

## MoE

## SwiGLUFFN

# Loss

## DPOLoss

## EntropyLoss

## GRPOLoss

## PPOLoss

## PretainLoss

## SFTLoss

# PEFT

## LoRALinear